# 3D Gaussian Splatting scan - free Colab T4 (proof of concept)

**What this is:** the *scan-pipeline proof of concept* for the farm digital
twin: photos -> 3DGS training -> `.ply` -> USD (`scan_io`) -> browser viewer.

**What this is NOT:** a farm scan. The capture is the INRIA public reference
scene (`truck` - outdoor, vegetation around it). Real field scans are the
post-hackathon Phase 3 item. Labeled honestly; see `docs/04` D2.

**Runtime:** `Runtime > Change runtime type > T4 GPU`. Time budget: install
~8 min, data ~3 min, training ~12-20 min (7k iters), render ~2 min.
**No COLMAP needed** - the dataset ships its SfM poses (`sparse/0`).


## 0. GPU check


In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun from the top"


## 1. 3DGS trainer (INRIA reference implementation)
Compiles three CUDA extensions (~6-10 min, once per session).


In [ ]:
!git clone --depth 1 https://github.com/graphdeco-inria/gaussian-splatting.git /content/gauss
%cd /content/gauss
!pip install -q plyfile tqdm
!pip install -q submodules/diff-gaussian-rasterization submodules/simple-knn submodules/fused-ssim
print("trainer ready")


## 2. Data: public reference capture (SfM poses included)
~650 MB download. The archive contains `tandt/truck`, `tandt/train`, `db/*`;
each scene ships `sparse/0` COLMAP models, so there is **no COLMAP step**.


In [ ]:
%cd /content
import os, subprocess
if not os.path.isdir("tandt/truck"):
    url = "https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip"
    subprocess.run(["wget", "-q", url], check=True)          # ~650 MB
    subprocess.run(["unzip", "-q", "tandt_db.zip"], check=True)
assert os.path.exists("tandt/truck/sparse/0/images.bin"), "COLMAP sparse model expected"
n = len(os.listdir("tandt/truck/images"))
print(f"truck: {n} images + COLMAP poses ready (no SfM needed)")

## 3. Train (7k iterations)
~12-20 min on T4. `--quiet` keeps the log small; PSNR lines are what we want.


In [ ]:
%cd /content/gauss
!python train.py -s /content/tandt/truck -m /content/out/truck --iterations 7000 --testing_iterations 3500 7000 --save_iterations 7000 --quiet 2>&1 | tee /content/train.log
print("training done")

## 4. Metrics


In [ ]:
import re, json
log = open("/content/train.log").read()
pairs = re.findall(r"\[ITER (\d+)\] Evaluating (\w+): PSNR ([\d.]+)", log)
metrics = {"psnr": {f"{it}_{split}": float(v) for it, split, v in pairs}}
print(json.dumps(metrics, indent=1))
json.dump(metrics, open("/content/metrics.json", "w"))

## 5. Flythrough video (novel views, real poses)
Renders every training camera in angular order around the scene center -
a guaranteed-good orbit without custom projection code. Fallback: skip this
cell, the PLY + viewer still demo fine.


In [ ]:
import shutil, os, sys, subprocess
# Clone the twin repo so scan_io is available, then produce the USD asset too.
if not os.path.isdir("/content/twin"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "pivot",
                    "https://github.com/Only1allan/3d_crop_Intelligence-v0.git", "/content/twin"], check=True)
sys.path.insert(0, "/content/twin/exts/omniverse_crop_intelligence")
ply = "/content/out/truck/point_cloud/iteration_7000/point_cloud.ply"
shutil.copy(ply, "/content/scan.ply")
from omniverse_crop_intelligence.scan_io import convert
summary = convert("/content/scan.ply", "/content/farm_scan.usda", max_points=150_000)
print("farm_scan.usda:", summary["points_out"], "points")
print("scan.ply:", round(os.path.getsize("/content/scan.ply") / 1e6, 1), "MB")
print("artifacts ready: /content/scan.ply, /content/farm_scan.usda, /content/orbit.mp4, /content/metrics.json")


## 6. Export artifacts
`scan.ply` -> upload to the twin notebook (step 7) and/or the GitHub web UI
(`web/viewer/scan.ply`) and/or [SuperSplat](https://supersplat.playcanvas.com)
for an instant hosted viewer. Keep files under 100 MB for the GitHub web UI;
if over, rerun training with `--iterations 5000` (fewer gaussians).


In [ ]:
import shutil, os, sys, subprocess
# Clone the twin repo so scan_io is available, then produce the USD asset too.
if not os.path.isdir("/content/twin"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "pivot",
                    "https://github.com/Only1allan/3d_crop_Intelligence-v0.git", "/content/twin"], check=True)
sys.path.insert(0, "/content/twin/exts/omniverse_crop_intelligence")
ply = "/content/out/truck/point_cloud/iteration_7000/point_cloud.ply"
shutil.copy(ply, "/content/scan.ply")
from omniverse_crop_intelligence.scan_io import convert
summary = convert("/content/scan.ply", "/content/farm_scan.usda", max_points=150_000)
print("farm_scan.usda:", summary["points_out"], "points")
print("scan.ply:", round(os.path.getsize("/content/scan.ply") / 1e6, 1), "MB")
print("artifacts ready: /content/scan.ply, /content/farm_scan.usda, /content/orbit.mp4, /content/metrics.json")


## Troubleshooting
| symptom | fix |
|---|---|
| CUDA OOM during training | rerun training with `--iterations 5000`; T4 fits this scene comfortably at 7k |
| submodule compile error | `Runtime > Disconnect and delete runtime`, reconnect T4, rerun from cell 1 |
| no GPU available in Colab | retry later, or use Kaggle (P100/T4x2, internet ON in settings) and adapt paths |
| download stalls | the dataset is re-fetchable; rerun cell 2 |
